In [1]:
# Name the project
TOPIC = "hela_0901"

In [3]:
# Set these variables to match your dataset and preferred annotator
PREFERRED_ANNOTATOR = "kellybillings"

In [ ]:
# %cd ..

In [ ]:
# %pip install loguru

In [ ]:
import os
import glob

from dotenv import load_dotenv
from sagemaker.session import Session
from sagemaker.tuner import (
    HyperparameterTuner,
    ContinuousParameter,
    IntegerParameter,
    CategoricalParameter,
)
from sagemaker.huggingface import HuggingFace

from dataloader.preprocess import (
    extract_span_tags_metadata,
    tags_metadata_to_id2label_mapping,
    process_tsv_export_for_idea_detection,
    kfold_validation_split_for_idea_detection,
)
from Notebooks.utils import unzip

# get env vars
load_dotenv()

# aws vars
sess = Session(default_bucket=os.getenv("BUCKET"))
aws_role = sess.get_caller_identity_arn()

### Process data export from Inception

In [ ]:
raw_zip = f"datasets/raw/{TOPIC}/{TOPIC}.zip"
raw_dir = f"datasets/raw/{TOPIC}"
processed_data_output_path = f"datasets/processed/{TOPIC}"
cv_splits_path = f"{processed_data_output_path}/splits"
os.makedirs(processed_data_output_path, exist_ok=True)
os.makedirs(cv_splits_path, exist_ok=True)
unzip(raw_zip, raw_dir)

In [ ]:
tag_df = extract_span_tags_metadata(
    export_dir=raw_dir,
    output_dir=raw_dir
)
id2label = tags_metadata_to_id2label_mapping(
    tag_metadata_path=raw_dir,
    output_dir=processed_data_output_path
)
df = process_tsv_export_for_idea_detection(
    export_dir=raw_dir,
    preferred_annotator=PREFERRED_ANNOTATOR,
    output_dir=processed_data_output_path
)

In [ ]:
splits = kfold_validation_split_for_idea_detection(
    df, n_splits=int(os.getenv("K_FOLDS")),  random_state=int(os.getenv("SEED"))
)
for data_split, df in splits.items():
    split_path = f"{processed_data_output_path}/splits/{data_split}.csv"
    df.to_csv(split_path, index=False)

### Upload to s3 bucket

In [ ]:
# Upload processed data to S3
sess.upload_data(
    path=f"{processed_data_output_path}/all.csv",
    bucket=os.getenv("BUCKET"),
    key_prefix=f"{os.getenv('IDEA_PREFIX')}/data/{TOPIC}",
)

In [ ]:
# Upload cv splits to S3
for csv_file in glob.glob(f"{cv_splits_path}/*.csv"):
    sess.upload_data(
        path=csv_file,
        bucket=os.getenv("BUCKET"),
        key_prefix=f"{os.getenv('IDEA_PREFIX')}/data/{TOPIC}/splits",
    )

### Hyperparameter sweep

In [ ]:
## HP tuning setup
metric_definitions = [
    {"Name": "CV_F1_MICRO", "Regex": "'cv_avg_eval_f1_micro': ([0-9\\.]+)"},
    {"Name": "CV_F1_MACRO", "Regex": "'cv_avg_eval_f1_macro': ([0-9\\.]+)"},
    {
        "Name": "CV_PRECISION_MICRO",
        "Regex": "'cv_avg_eval_precision_micro': ([0-9\\.]+)",
    },
    {
        "Name": "CV_PRECISION_MACRO",
        "Regex": "'cv_avg_eval_precision_macro': ([0-9\\.]+)",
    },
    {"Name": "CV_RECALL_MICRO", "Regex": "'cv_avg_eval_recall_micro': ([0-9\\.]+)"},
    {"Name": "CV_RECALL_MACRO", "Regex": "'cv_avg_eval_recall_macro': ([0-9\\.]+)"},
]
hyperparameter_ranges = {
    "learning_rate": ContinuousParameter(0.00001, 0.0001, scaling_type="Logarithmic"),
    "warmup_steps": IntegerParameter(8, 65),
    "weight_decay": ContinuousParameter(0.01, 0.1, scaling_type="Logarithmic"),
    "lr_scheduler_type": CategoricalParameter(["linear", "cosine"]),
    "model_name": CategoricalParameter(
        [
            "allenai/scibert_scivocab_uncased",
            "google-bert/bert-base-uncased",
            "microsoft/deberta-v3-base",
        ]
    ),
}
static_hyperparameters = {
    "project_name": f"{TOPIC}-idea-detection",
    "epochs": 30,
    "train_batch_size": 16,
    "eval_batch_size": 32,
    "cv": True,
    "n_folds": int(os.getenv("K_FOLDS")),
    # "n_folds": 3,
    "wandb_api_key": os.getenv("WANDB_API_KEY"),
}
cv_estimator = HuggingFace(
    source_dir=".",
    entry_point="src/train.py",
    hyperparameters=static_hyperparameters,
    code_location=os.getenv("KI_SCRIPT_PATH"),
    instance_type=os.getenv("TRAIN_INSTANCE_TYPE"),
    instance_count=1,
    role=aws_role,
    image_uri=os.getenv("TRAIN_DOCKER_IMAGE"),
    py_version="py310",
    use_spot_instances=True,
    max_wait=7200,
    max_run=4000,
    output_path=os.getenv("KI_MODEL_PATH"),
)
cv_tuner = HyperparameterTuner(
    estimator=cv_estimator,
    base_tuning_job_name=f"{TOPIC.replace('_', '-')}-ki-cv-sweep",
    hyperparameter_ranges=hyperparameter_ranges,
    metric_definitions=metric_definitions,
    objective_metric_name="CV_F1_MACRO",
    objective_type="Maximize",
    max_jobs=10,
)
cv_tuner.fit(
    {
        "model_dir": f"{os.getenv('IDEA_MODEL_PATH')}",
        "dataset_path": f"s3://{os.getenv('BUCKET')}/{os.getenv('IDEA_PREFIX')}/data/{TOPIC}",
        "cv_data": f"s3://{os.getenv('BUCKET')}/{os.getenv('IDEA_PREFIX')}/data/{TOPIC}/splits",
    }
)

In [ ]:
hyperparameters = {
    # training parameters
    "epochs": 15,
    "learning_rate": 8.8942e-5,
    "warmup_steps": 29,
    "train_batch_size": 16,
    "gradient_accumulation_steps": 2,
    "eval_batch_size": 32,
    "model_name": "google-bert/bert-base-uncased",
    "weight_decay": 0.014,
    "lr_scheduler_type": "cosine",
    # paths and env variables
    "project_name": f"{TOPIC}-idea-detection",
    "wandb_api_key": os.getenv("WANDB_API_KEY")
}

idea_estimator = HuggingFace(
    source_dir=".",
    entry_point="src/train.py",
    hyperparameters=hyperparameters,
    code_location=os.getenv("IDEA_SCRIPT_PATH"),
    instance_type=os.getenv("TRAIN_INSTANCE_TYPE"),
    instance_count=1,
    role=aws_role,
    image_uri=os.getenv("TRAIN_DOCKER_IMAGE"),
    py_version="py310",
    use_spot_instances=True,
    max_wait=7200,  # max time including spot start + training time
    max_run=4000,  # expected training time
    output_path=os.getenv("IDEA_MODEL_PATH"),
)
idea_estimator.fit(
    {
        "dataset_path": f"{os.getenv('KI_DATA_PATH')}/{TOPIC}",
        "model_dir": f"{os.getenv('IDEA_MODEL_PATH')}",
    }
)